# Air-Gap Geocoding Stack - Demonstration Notebook

A comprehensive walkthrough of the `airgap_geocoding` Python library running against the
self-hosted Docker services. All HTTP traffic stays on localhost - no external APIs are
called.

## Prerequisites

The notebook requires four self-hosted services:

| Service                       | Default port | Docker Compose path |
| ----------------------------- | ------------ | ------------------- |
| Nominatim (forward geocoding) | 8080         | `docker/nominatim/` |
| Photon (reverse geocoding)    | 2322         | `docker/photon/`    |
| OSRM + HAProxy (routing)      | 80           | `docker/osrm/`      |
| postcodes.io                  | 8000         | `docker/osrm/`      |

Either start the services manually by following the `README.md` in each `docker/`
subdirectory, or use the **Start Services** cell immediately below to launch all three
stacks from within this notebook.

Endpoint URLs are controlled by four environment variables in `.env` (see `.env.example`).


______________________________________________________________________

## Start Services (optional)

Run the cell below to start the combined stack (`docker/docker-compose.yml`) and wait
until all services respond. **Skip this section if the services are already running.**

Data volumes must be prepared before containers can serve requests. If this is a fresh
deployment, consult the README for each service first:

- `docker/nominatim/README.md` - OSM PBF import (may take 10-30 minutes on first run)
- `docker/photon/README.md` - Photon Elasticsearch data bake
- `docker/osrm/README.md` - OSRM extract / partition / customise per profile

The combined compose file requires `OSRM_DATA` to be set in the project root `.env`
(see `.env.example`).


In [ ]:
import pathlib
import subprocess
import time

import requests as _requests

from airgap_geocoding.settings import NOMINATIM_URL, OSRM_API, PHOTON_API, POSTCODES_URL


def _find_repo_root(start: pathlib.Path) -> pathlib.Path:
    """Walk up the directory tree until pyproject.toml is found (repo root marker)."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


_REPO_ROOT = _find_repo_root(pathlib.Path().resolve())
_COMPOSE_FILE = _REPO_ROOT / "docker" / "docker-compose.yml"
_ENV_FILE = _REPO_ROOT / ".env"

_HEALTH_TIMEOUT = 120  # seconds to wait for all services to respond

_ENDPOINTS = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
    "OSRM / HAProxy": OSRM_API,
    "postcodes.io": POSTCODES_URL,
}


def start_services(timeout: int = _HEALTH_TIMEOUT) -> None:
    """Start the combined Docker Compose stack and poll until every endpoint responds."""
    print(f"Starting stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "up",
            "-d",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    ready = {name: False for name in _ENDPOINTS}
    deadline = time.monotonic() + timeout
    print(f"\nPolling services (timeout {timeout}s) ...")

    while time.monotonic() < deadline:
        for name, url in _ENDPOINTS.items():
            if ready[name]:
                continue
            try:
                _requests.get(url, timeout=3)
                ready[name] = True
                print(f"  ✓  {name} is up  ({url})")
            except Exception:
                pass
        if all(ready.values()):
            break
        time.sleep(3)

    still_down = [n for n, ok in ready.items() if not ok]
    if still_down:
        print(f"\n[WARNING] Timed out waiting for: {', '.join(still_down)}")
        print("Verify data volumes are prepared and check container logs:")
        print(f"  docker compose -f {_COMPOSE_FILE.relative_to(_REPO_ROOT)} logs")
    else:
        print("\nAll services are up and ready.")


start_services()

## Setup - Imports and Configuration


In [ ]:
import importlib
import pprint

import folium
import pandas as pd
import requests

# Re-read settings in case .env was updated after the kernel started
import airgap_geocoding.settings as _settings

importlib.reload(_settings)

from airgap_geocoding import (  # noqa: E402
    geocoder,
    lookup_outcode,
    lookup_postcode,
    route,
)
from airgap_geocoding.settings import (  # noqa: E402
    NOMINATIM_URL,
    OSRM_API,
    PHOTON_API,
    POSTCODES_URL,
)

print("Configured service endpoints")
print(f"  Nominatim (forward geocoding) : {NOMINATIM_URL}")
print(f"  Photon    (reverse geocoding) : {PHOTON_API}")
print(f"  OSRM / HAProxy (routing)      : {OSRM_API}")
print(f"  postcodes.io                  : {POSTCODES_URL}")

______________________________________________________________________

## 1 - Service Health Checks

Before running the demo, confirm that every service is reachable. A `✓` indicates an HTTP
response was received (any status); `✗` means the connection failed entirely.


In [ ]:
_SERVICES = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
    "OSRM / HAProxy": OSRM_API,
    "postcodes.io": POSTCODES_URL,
}

rows = []
for name, base_url in _SERVICES.items():
    try:
        r = requests.get(base_url, timeout=5)
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": r.status_code,
                "Reachable": "✓",
            }
        )
    except Exception:
        rows.append(
            {"Service": name, "URL": base_url, "HTTP Status": "ERR", "Reachable": "✗"}
        )

pd.DataFrame(rows).set_index("Service")

______________________________________________________________________

## 2 - Forward Geocoding (Nominatim)

`geocoder(location)` accepts a free-text place name or postcode. When the input is not a
valid `lat,lon` pair it forwards the query to **Nominatim** (`/search`), then enriches the
result with a reverse-geocode call to **Photon** to obtain a structured GeoJSON feature.

The return value always contains a top-level `geo` key: `{"lat": float, "lon": float}`.


In [ ]:
# Forward geocode an address
result_address = geocoder("10 Downing Street, London")
pprint.pprint(result_address)

In [ ]:
# A postcode string also resolves via Nominatim
result_postcode_geocode = geocoder("EC2V 6DN")  # Bank of England
geo = result_postcode_geocode.get("geo", {})
print(f"EC2V 6DN  →  lat={geo.get('lat')}, lon={geo.get('lon')}")

In [ ]:
# Map the result - Nominatim result for 10 Downing Street
geo = result_address.get("geo", {})
lat = geo.get("lat", 51.5034)
lon = geo.get("lon", -0.1276)

props = result_address.get("properties", {})
label = props.get("name") or props.get("street") or "10 Downing Street"

m_fwd = folium.Map(location=[lat, lon], zoom_start=16)
folium.Marker(
    [lat, lon],
    popup=folium.Popup(
        f"<b>{label}</b><br>lat={lat:.5f}, lon={lon:.5f}", max_width=250
    ),
    tooltip="Forward geocoding result",
    icon=folium.Icon(color="blue", icon="home"),
).add_to(m_fwd)
m_fwd

______________________________________________________________________

## 3 - Reverse Geocoding (Photon)

When the input to `geocoder()` is a valid `"lat,lon"` string, the coordinate is sent
directly to **Photon** (`/reverse`) without consulting Nominatim. Photon returns a GeoJSON
feature whose `properties` include name, street, city, and country fields.

Photon is backed by a pre-built Elasticsearch index derived from OpenStreetMap data.


In [ ]:
# Reverse geocode Parliament Square
result_rev = geocoder("51.5007, -0.1246")
pprint.pprint(result_rev)

In [ ]:
# Display key properties
props = result_rev.get("properties", {})
geo_rev = result_rev.get("geo", {})

print(f"Coordinates  : {geo_rev.get('lat')}, {geo_rev.get('lon')}")
print(f"Name         : {props.get('name', '-')}")
print(f"Street       : {props.get('street', '-')}")
print(f"City         : {props.get('city', '-')}")
print(f"Country code : {props.get('countrycode', '-')}")

In [ ]:
# Map the reverse geocoding result
lat_r = geo_rev.get("lat", 51.5007)
lon_r = geo_rev.get("lon", -0.1246)
name_r = props.get("name") or "Parliament Square"

m_rev = folium.Map(location=[lat_r, lon_r], zoom_start=16)
folium.Marker(
    [lat_r, lon_r],
    popup=folium.Popup(
        f"<b>{name_r}</b><br>lat={lat_r:.5f}, lon={lon_r:.5f}", max_width=250
    ),
    tooltip="Reverse geocoding result",
    icon=folium.Icon(color="green", icon="map-marker"),
).add_to(m_rev)
m_rev

______________________________________________________________________

## 4 - Routing (OSRM via HAProxy)

`route(origin, destination, profile)` calls **OSRM** through the HAProxy load-balancer.
HAProxy routes requests to the appropriate OSRM backend based on the profile path prefix:

| Profile   | Backend container | Typical use              |
| --------- | ----------------- | ------------------------ |
| `driving` | `osrm-driving`    | Car routes, fastest path |
| `walking` | `osrm-foot`       | Pedestrian routes        |
| `cycling` | `osrm-bike`       | Bicycle routes           |

Both `origin` and `destination` are `(lat, lon)` tuples. The raw OSRM JSON is returned;
the route `geometry` is an encoded polyline string.


In [ ]:
# Geocode origin and destination
result_origin = geocoder("King's Cross Station, London")
result_dest = geocoder("Victoria Station, London")

origin_coords = (result_origin["geo"]["lat"], result_origin["geo"]["lon"])
dest_coords = (result_dest["geo"]["lat"], result_dest["geo"]["lon"])

print(f"Origin      : King's Cross  → {origin_coords}")
print(f"Destination : Victoria      → {dest_coords}")

In [ ]:
# Route using all three profiles and compare
_PROFILES = ["driving", "walking", "cycling"]
route_results = {}
rows = []

for profile in _PROFILES:
    r = route(origin_coords, dest_coords, profile=profile)
    route_results[profile] = r
    if r and r.get("routes"):
        leg = r["routes"][0]
        rows.append(
            {
                "Profile": profile.capitalize(),
                "Distance (km)": round(leg["distance"] / 1000, 2),
                "Duration (min)": round(leg["duration"] / 60, 1),
            }
        )
    else:
        rows.append(
            {
                "Profile": profile.capitalize(),
                "Distance (km)": "N/A",
                "Duration (min)": "N/A",
            }
        )

pd.DataFrame(rows).set_index("Profile")

In [ ]:
# Demonstrate that an unsupported profile raises ValueError
try:
    route(origin_coords, dest_coords, profile="flying")
except ValueError as exc:
    print(f"ValueError raised as expected:\n  {exc}")

In [ ]:
def _decode_polyline(encoded: str) -> list[tuple[float, float]]:
    """Decode an OSRM / Google encoded polyline string into (lat, lon) pairs."""
    coords: list[tuple[float, float]] = []
    index = 0
    lat = 0
    lng = 0
    while index < len(encoded):
        for is_lng in (False, True):
            shift, result = 0, 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1F) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = ~(result >> 1) if (result & 1) else (result >> 1)
            if is_lng:
                lng += delta
            else:
                lat += delta
        coords.append((lat / 1e5, lng / 1e5))
    return coords


# Map the driving route
driving_data = route_results.get("driving", {})
centre_lat = (origin_coords[0] + dest_coords[0]) / 2
centre_lon = (origin_coords[1] + dest_coords[1]) / 2

m_route = folium.Map(location=[centre_lat, centre_lon], zoom_start=13)

# Origin and destination markers
folium.Marker(
    list(origin_coords),
    tooltip="King's Cross (origin)",
    icon=folium.Icon(color="green", icon="play"),
).add_to(m_route)
folium.Marker(
    list(dest_coords),
    tooltip="Victoria (destination)",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m_route)

# Decode and draw the polyline
if driving_data and driving_data.get("routes"):
    encoded = driving_data["routes"][0].get("geometry", "")
    if encoded:
        decoded = _decode_polyline(encoded)
        folium.PolyLine(
            decoded, color="royalblue", weight=4, tooltip="Driving route"
        ).add_to(m_route)

m_route

______________________________________________________________________

## 5 - Postcode Lookup (postcodes.io)

Two helpers wrap the self-hosted [postcodes.io](https://postcodes.io/) API:

- `lookup_postcode(postcode)` - full UK postcode, e.g. `"SW1A 2AA"`
- `lookup_outcode(outcode)` - outward code (district), e.g. `"SW1A"`

Both return the `result` object from the API response, or an empty dict on failure.


In [ ]:
# Full postcode lookup
pc_result = lookup_postcode("SW1A 2AA")
pprint.pprint(pc_result)

In [ ]:
# Outcode lookup
oc_result = lookup_outcode("SW1A")
pprint.pprint(oc_result)

In [ ]:
# Summary table of key postcode fields
_PC_FIELDS = [
    "postcode",
    "latitude",
    "longitude",
    "admin_district",
    "admin_ward",
    "parliamentary_constituency",
    "ccg",
    "nuts",
]

rows = []
for field in _PC_FIELDS:
    rows.append(
        {
            "Field": field,
            "SW1A 2AA": pc_result.get(field, "-"),
            "SW1A (outcode)": oc_result.get(field, "-"),
        }
    )

pd.DataFrame(rows).set_index("Field")

______________________________________________________________________

## 6 - Combined Workflow

End-to-end scenario: given two UK postcodes, geocode both locations, calculate driving and
walking routes between them, and render both on a single map with the postcode metadata
shown in the marker popups.


In [ ]:
# --- Inputs -----------------------------------------------------------
POSTCODE_A = "EC2V 6DN"  # Bank of England
POSTCODE_B = "WC2N 5DU"  # Trafalgar Square

# --- Step 1: postcode metadata ----------------------------------------
pc_a = lookup_postcode(POSTCODE_A)
pc_b = lookup_postcode(POSTCODE_B)

# --- Step 2: geocode --------------------------------------------------
geo_a = geocoder(POSTCODE_A)
geo_b = geocoder(POSTCODE_B)

coords_a = (geo_a["geo"]["lat"], geo_a["geo"]["lon"])
coords_b = (geo_b["geo"]["lat"], geo_b["geo"]["lon"])

print(f"{POSTCODE_A}  →  {coords_a}  ({pc_a.get('admin_district', '?')})")
print(f"{POSTCODE_B}  →  {coords_b}  ({pc_b.get('admin_district', '?')})")

# --- Step 3: routing --------------------------------------------------
route_driving = route(coords_a, coords_b, profile="driving")
route_walking = route(coords_a, coords_b, profile="walking")


def _route_summary(r: dict, label: str) -> None:
    if r and r.get("routes"):
        leg = r["routes"][0]
        print(
            f"{label:10s}  distance={leg['distance'] / 1000:.2f} km  duration={leg['duration'] / 60:.1f} min"
        )
    else:
        print(f"{label:10s}  no route returned")


_route_summary(route_driving, "driving")
_route_summary(route_walking, "walking")

In [ ]:
# --- Step 4: combined map ---------------------------------------------
mid_lat = (coords_a[0] + coords_b[0]) / 2
mid_lon = (coords_a[1] + coords_b[1]) / 2
m_combined = folium.Map(location=[mid_lat, mid_lon], zoom_start=14)

# Postcode markers with metadata popups
for coords, pc_data, label, colour in [
    (coords_a, pc_a, POSTCODE_A, "blue"),
    (coords_b, pc_b, POSTCODE_B, "red"),
]:
    popup_html = (
        f"<b>{label}</b><br>"
        f"District: {pc_data.get('admin_district', '-')}<br>"
        f"Ward: {pc_data.get('admin_ward', '-')}<br>"
        f"Constituency: {pc_data.get('parliamentary_constituency', '-')}"
    )
    folium.Marker(
        list(coords),
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=label,
        icon=folium.Icon(color=colour, icon="info-sign"),
    ).add_to(m_combined)

# Draw route polylines
_ROUTE_STYLES = {
    "driving": (route_driving, "royalblue", "Driving route"),
    "walking": (route_walking, "darkorange", "Walking route"),
}
for _profile, (r_data, colour, tip) in _ROUTE_STYLES.items():
    if r_data and r_data.get("routes"):
        encoded = r_data["routes"][0].get("geometry", "")
        if encoded:
            folium.PolyLine(
                _decode_polyline(encoded),
                color=colour,
                weight=4,
                opacity=0.8,
                tooltip=tip,
            ).add_to(m_combined)

m_combined

______________________________________________________________________

## 7 - Error Handling Showcase

All library functions follow a consistent defensive pattern:

- **Non-200 HTTP response** → return `{}`
- **Connection error / timeout** → `except Exception` swallows and returns `{}`
- **Invalid routing profile** → `ValueError` (the one deliberate exception)

The cells below demonstrate each case without relying on mocked HTTP.


In [ ]:
import os
from unittest.mock import patch

# --- Case 1: unreachable service (bad port) ---------------------------
# Temporarily override the Photon URL to a port nothing listens on
with patch("airgap_geocoding.geocoding.PHOTON_API", "http://localhost:19999"):
    bad_rev = __import__(
        "airgap_geocoding.geocoding", fromlist=["photon_reverse_geocode"]
    ).photon_reverse_geocode(51.5, -0.1)
print(f"Unreachable Photon → {bad_rev!r}  (expected: {{}})")

# --- Case 2: empty Nominatim results (nonsense query) -----------------
empty_result = geocoder("xyzzy_this_place_does_not_exist_anywhere_12345")
print(f"Unknown place     → {empty_result!r}  (expected: {{}})")

# --- Case 3: invalid routing profile raises ValueError ----------------
try:
    route((51.5, -0.1), (52.5, -1.9), profile="teleportation")
except ValueError as exc:
    print(f"Bad profile       → ValueError: {exc}")

In [ ]:
# --- Case 4: bad postcode returns empty dict ---------------------------
bad_pc = lookup_postcode("ZZ9 9ZZ")  # Invalid postcode
print(f"Invalid postcode  → {bad_pc!r}  (expected: {{}})")

bad_oc = lookup_outcode("ZZ9")  # Invalid outcode
print(f"Invalid outcode   → {bad_oc!r}  (expected: {{}})")

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers for the three stacks.
Persistent data volumes are **not** removed, so restarting later will be fast.


In [ ]:
def stop_services() -> None:
    """Stop the combined Docker Compose stack. Data volumes are preserved."""
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()